# Bronze Layer: Reverse Historical Ingestion
This notebook fetches seismic data from the USGS API year by year, running in **reverse chronological order** (from 2026 back to 2000). 
It writes raw JSONs into DBFS storage and logs the metadata into our `bronze.seismic_raw` table.

# 🏗️ Bronze Layer Architectural Design Decisions

---

### 1. Raw Data Preservation & Dual-Storage Footprint
> **The Decision:** 
> We store the entire, unaltered GeoJSON response as a single raw string (`raw_payload`) in our Delta table (`bronze.seismic_raw`), alongside a native file copy (`.json`) saved directly into a Unity Catalog Managed Volume (`/Volumes/cr_seismic_analysis/bronze/seismic_data_raw/`).
> 
> **The "Why":** 
> APIs are dynamic and subject to structural shifts. By keeping the raw payload intact, we protect our ingestion from sudden API updates. If we need to extract new fields in the future, or if we discover a bug in our parsing logic, we can re-run our downstream **Silver Layer** parsing scripts over historical data without making a single external API call. Using Managed Volumes instead of legacy DBFS provides native governance, auditing, and POSIX-compliant file paths.

---

### 2. Dual-Layer Idempotency & Ingestion Strategy
> **The Decision:** 
> We check if a static historical file exists in our Managed Volume before requesting it from the API to skip duplicate work. To ensure transactional integrity inside the Delta table, we run a targeted SQL `DELETE` command matching the specific `source_file` name immediately before executing a PySpark `mode("append")` write.
> 
> **The "Why":** 
> This ensures that if this notebook is run multiple times—either manually or by an orchestrator—it will never duplicate rows in our database. It allows us to safely re-run the current dynamic calendar year (2026) to append new tremors without corrupting or duplicating our 26-year historical baseline.

---

### 3. Separation of Concerns (Schema-on-Read)
> **The Decision:** 
> This script is dedicated solely to **Ingestion** (moving data from the source into our ecosystem). Parsing, casting, flattening, and data validation are decoupled and deferred entirely to the **Silver Layer**.
> 
> **The "Why":** 
> Keeping the Bronze layer purely about data movement makes the initial pipeline highly resilient, simple to debug, and incredibly fast. It leaves the complex task of unpacking nested JSON structures to Spark's distributed cluster engine downstream.

---

### 4. API Politeness (Rate Limiting)
> **The Decision:** 
> We integrate a mandatory `time.sleep(1.5)` throttle between yearly iterations.
> 
> **The "Why":** 
> Public scientific APIs (like the USGS) can temporarily throttle or block Databricks cluster IPs if hit with rapid, concurrent requests. A slight courtesy delay ensures a stable, stable, uninterrupted historical harvest.

In [0]:
import requests
import json
import time  
import os
from datetime import datetime
from pyspark.sql import functions as F

# Set catalog and schema context
spark.sql("USE CATALOG cr_seismic_analysis")
spark.sql("USE SCHEMA bronze")

# 1. Dynamic Parameter Configuration
CURRENT_YEAR = datetime.now().year
START_YEAR = 2000
END_YEAR = CURRENT_YEAR  # Automatically adjusts to the current year
BASE_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

# Path targeting the Unity Catalog Managed Volume
BRONZE_MOUNT_PATH = "/Volumes/cr_seismic_analysis/bronze/seismic_data_raw/"

# 2. Reverse Chronological Ingestion Loop
for year in range(END_YEAR, START_YEAR - 1, -1):
    file_name = f"seismic_raw_{year}.json"
    dest_path = os.path.join(BRONZE_MOUNT_PATH, file_name)
    
    # Skip API requests for static historical years if the file already exists in the Volume
    if year != CURRENT_YEAR and os.path.exists(dest_path):
        print(f"Year {year} already ingested. Skipping to avoid duplicate request.")
        continue
        
    print(f"Downloading data for Year {year}...")
    
    params = {
        "format": "geojson",
        "starttime": f"{year}-01-01",
        "endtime": f"{year}-12-31",
        "minlatitude": 8.0,
        "maxlatitude": 11.5,
        "minlongitude": -86.0,
        "maxlongitude": -82.5
    }
    
    try:
        response = requests.get(BASE_URL, params=params, timeout=30)
        
        if response.status_code == 200:
            json_data = response.json()
            
            # Save the raw JSON payload to the Volume (overwriting current year with fresh data)
            with open(dest_path, "w") as f:
                json.dump(json_data, f)
            
            raw_payload_str = json.dumps(json_data)
            current_time = datetime.now()
            
            # Create the Spark DataFrame for the current batch
            df_batch = spark.createDataFrame(
                [(raw_payload_str, current_time, file_name)],
                schema=["raw_payload", "ingestion_timestamp", "source_file"]
            )
            
            # Native Delta Lake MERGE execution for absolute idempotency
            target_table = "seismic_raw"
            from delta.tables import DeltaTable
            
            delta_target = DeltaTable.forName(spark, f"cr_seismic_analysis.bronze.{target_table}")
            
            delta_target.alias("target") \
                .merge(
                    df_batch.alias("source"),
                    "target.source_file = source.source_file"
                ) \
                .whenMatchedUpdate(set={
                    "raw_payload": "source.raw_payload",
                    "ingestion_timestamp": "source.ingestion_timestamp"
                }) \
                .whenNotMatchedInsert(values={
                    "raw_payload": "source.raw_payload",
                    "ingestion_timestamp": "source.ingestion_timestamp",
                    "source_file": "source.source_file"
                }) \
                .execute()

            print(f"Successfully merged Year {year} with {len(json_data.get('features', []))} events.")
            
        else:
            print(f"API returned HTTP error {response.status_code} for year {year}.")
            
    except Exception as e:
        print(f"An error occurred while fetching year {year}: {str(e)}")
        
    print("Waiting 1.5 seconds before next request...")
    time.sleep(1.5)

In [0]:
%sql
-- Verification of the files in the bronze layer
USE CATALOG cr_seismic_analysis;

-- List files using the explicit string path
LIST '/Volumes/cr_seismic_analysis/bronze/seismic_data_raw/';

In [0]:
%sql
-- Verification of the files in the bronze layer
USE CATALOG cr_seismic_analysis;

-- Check total file count and unique years captured in the Bronze table
SELECT 
    COUNT(*) as total_ingested_files,
    COUNT(DISTINCT source_file) as unique_years_count
FROM bronze.seismic_raw;